# Dialog Chrome, Navigation, Launcher & Accessibility (`CRAFT.js`, `CRAFT.css`)

> Page-level UI chrome around Solveit dialogs: mobile/desktop mode detection (`window.craftMobile`), auto-hiding top navbar, compact Monaco line height, `Mod+Shift+P` command/file launcher, auto-hiding idle composer (`window.craftIdleComposer`), and tooltip-to-`aria-label` accessibility pass (`window.craftA11y`).

`window.craftMobile` runs here early in `CRAFT.js` so `html.craft-mobile` is applied before cell presentation, insertion, and editor modules mount.

## Chrome Templates (`#| export_ft`)

Icons and static DOM shells for the mobile/desktop view toggle (`#craft-mobile-toggle`) and the command launcher modal (`#solveit-launcher`).

In [ ]:
#| export_ft CRAFT.js
#| eval: false
TEMPLATES = dict(
    mobile_phone_icon=svg_icon([ft('rect', x='7', y='2', width='10', height='20', rx='2'), 'M11 18h2'], size=15, sw=1.8),
    mobile_desk_icon=svg_icon([ft('rect', x='2', y='3', width='20', height='14', rx='2'), 'M8 21h8M12 17v4'], size=15, sw=1.8),
    mobile_toggle_btn=Button(id='craft-mobile-toggle', klass='craft-nav-icon-btn', type='button'),
    launcher_root=craft_overlay(
        craft_card(
            Input(type='text', placeholder='Run a command or open a dialog/file…',
                  autocomplete='off', spellcheck='false', aria_label='Search commands and files'),
            Ul(role='listbox'),
            cls='solveit-launcher-card'),
        id='solveit-launcher', role='dialog', aria_modal='true', aria_label='Command palette'),
)

## Mobile Mode Detection (`window.craftMobile`)

Auto-enables `html.craft-mobile` on touch-only devices (`(hover: none) and (pointer: coarse)`) while letting desktop users toggle mobile view explicitly via `#craft-mobile-toggle`, the launcher, or `window.craftMobile.setMode('auto' | 'mobile' | 'desktop')`.

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Mobile mode detection & view toggle ───────────────────────────────
   Auto-enables `html.craft-mobile` only on touch/mobile devices (`(hover: none)
   and (pointer: coarse)`), so shrinking a desktop window stays in desktop view.
   Users can explicitly switch between Desktop and Mobile at any time via the
   navbar toggle button (#craft-mobile-toggle), the launcher, or
   `window.craftMobile.setMode('auto'|'mobile'|'desktop')`. */
(() => {
  window.craftMobile?.disable?.();
  const {mountTpl, qs, on, store, patch, onDomChange} = window.__craftCMCore;
  const ac = new AbortController(), sig = {signal: ac.signal};
  const PREF_KEY = 'craft-mobile-mode';
  const mq = matchMedia('(hover: none) and (pointer: coarse)');
  const root = document.documentElement, listeners = new Set();
  let meta = qs('meta[name="viewport"]'), addedMeta = false, prevMeta = meta?.content;
  const VIEWPORT = 'width=device-width, initial-scale=1, viewport-fit=cover, interactive-widget=resizes-content';
  if (!meta) { meta = document.createElement('meta'); meta.name = 'viewport'; document.head.append(meta); addedMeta = true; }
  if (!/width\s*=\s*device-width/.test(meta.content || '')) meta.content = VIEWPORT;
  else {
    if (!/viewport-fit/.test(meta.content)) meta.content += ', viewport-fit=cover';
    if (!/interactive-widget/.test(meta.content)) meta.content += ', interactive-widget=resizes-content';
  }
  const iOS = /iP(hone|ad|od)/.test(navigator.userAgent) || (navigator.platform === 'MacIntel' && navigator.maxTouchPoints > 1);
  if (iOS && !/maximum-scale/.test(meta.content)) meta.content += ', maximum-scale=1';

  const readPref = () => { const v = store.get(PREF_KEY); return (v === 'mobile' || v === 'desktop') ? v : 'auto'; };
  let pref = readPref();
  const computeActive = () => pref === 'mobile' ? true : pref === 'desktop' ? false : mq.matches;

  const TPL = window.__craftTpl || {};
  const PHONE_SVG = TPL.mobile_phone_icon || '', DESK_SVG = TPL.mobile_desk_icon || '';

  const toggleBtn = mountTpl('mobile_toggle_btn');

  const syncToggleBtn = () => {
    const isMob = root.classList.contains('craft-mobile');
    const key = `${isMob ? 'm' : 'd'}:${pref}`;
    if (toggleBtn.dataset.state === key) return;
    const title = isMob
      ? `Switch to desktop view (current: ${pref === 'auto' ? 'auto mobile' : 'mobile'})`
      : `Switch to mobile view (current: ${pref === 'auto' ? 'auto desktop' : 'desktop'})`;
    patch(toggleBtn, {dataset: {state: key}, innerHTML: isMob ? DESK_SVG : PHONE_SVG, title, 'aria-label': title, 'aria-pressed': String(isMob)});
  };

  const positionToggleBtn = () => {
    if (root.classList.contains('craft-mobile')) return;
    if (toggleBtn.style.left !== 'auto' || toggleBtn.style.right !== '10px' || toggleBtn.style.top !== '6px') Object.assign(toggleBtn.style, { left: 'auto', right: '10px', top: '6px' });
  };

  const mountToggleBtn = () => {
    if (!toggleBtn.isConnected && document.body) document.body.append(toggleBtn);
    syncToggleBtn();
    positionToggleBtn();
  };

  const apply = () => {
    const on = computeActive();
    const changed = root.classList.contains('craft-mobile') !== on;
    if (changed) {
      root.classList.toggle('craft-mobile', on);
      listeners.forEach(fn => { try { fn(on); } catch (err) { console.error('craftMobile listener', err); } });
    }
    syncToggleBtn();
  };

  const setMode = m => {
    pref = (m === 'mobile' || m === 'desktop') ? m : 'auto';
    store.set(PREF_KEY, pref === 'auto' ? null : pref);
    apply();
    return root.classList.contains('craft-mobile');
  };
  const toggle = () => setMode(root.classList.contains('craft-mobile') ? 'desktop' : 'mobile');
  on(toggleBtn, 'click', e => { e.preventDefault(); e.stopPropagation(); toggle(); }, sig);
  on(mq, 'change', apply, sig);
  let offDom = null;
  on(window, 'resize', positionToggleBtn, {passive: true, signal: ac.signal});
  const start = () => {
    mountToggleBtn();
    offDom = onDomChange(mountToggleBtn, {delay: 'raf'});
  };
  apply();
  if (document.body) start(); else on(document, 'DOMContentLoaded', start, {once: true, signal: ac.signal});

  window.__hotkeyRegister?.({
    id: 'craft-toggle-mobile',
    label: 'Toggle mobile / desktop view',
    group: 'CRAFT · Layout',
    combo: '',
    handler: () => { toggle(); return true; },
  });

  window.craftMobile = {
    active: () => root.classList.contains('craft-mobile'),
    mode: () => pref,
    setMode,
    toggle,
    onChange: fn => { listeners.add(fn); return () => listeners.delete(fn); },
    reposition: positionToggleBtn,
    disable() {
      ac.abort();
      offDom?.();
      toggleBtn.remove();
      root.classList.remove('craft-mobile');
      if (addedMeta) meta.remove(); else if (prevMeta != null) meta.content = prevMeta;
      listeners.clear();
      delete window.craftMobile;
    },
  };
})();

## Auto-Hiding Top Nav

Keeps the navbar out of the way while reading and reveals it when the pointer enters a trigger band below the browser's top chrome, or when a navbar control has focus.

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Auto-hiding top nav ──────────────────────────────────────────────
   The reveal trigger is decoupled from the bar. The band starts BELOW the
   browser's own top-edge hotspot so a hidden browser toolbar doesn't fight
   it, and the bar stays pinned while the pointer is over the nav or inside
   an open UIkit drop, fading only after a grace period. */
(() => {
  window.craftAutoNav?.disable?.();
  const ac = new AbortController();
  const {qs, on, MODAL_OPEN_SEL} = window.__craftCMCore;
  const TRIGGER_TOP = 28;      // px: clear of the browser's top-edge hotspot
  const TRIGGER_BOTTOM = 76;   // px: bottom of the reveal band
  const HIDE_DELAY = 700;      // ms grace period before fading out

  const SEL = '.uk-container > nav.monster-navbar';
  const PIN = '.uk-drop, .uk-dropdown, [uk-drop], [uk-dropdown]';

  const nav = () => qs(SEL);
  let timer = null, last = null;

  function wantOpen() {
    if (qs(MODAL_OPEN_SEL)) return false;
    if (!last) return false;
    if (last.y >= TRIGGER_TOP && last.y <= TRIGGER_BOTTOM) return true;
    const t = last.target;
    if (t instanceof Element) {
      if (nav()?.contains(t)) return true;
      if (t.closest(PIN)) return true;
    }
    return !!qs('.uk-drop.uk-open, .uk-dropdown.uk-open');
  }

  function sync() {
    const n = nav();
    if (!n) return;
    if (wantOpen()) {
      clearTimeout(timer);
      timer = null;
      n.classList.add('nav-shown');
    } else if (!timer) {
      timer = setTimeout(() => {
        timer = null;
        if (!wantOpen()) nav()?.classList.remove('nav-shown');
      }, HIDE_DELAY);
    }
  }

  on(document, 'mousemove', e => {
    // Phones: taps emit compat mousemoves, which would flash the bar at random. The mobile toolbar's ☰ drives it instead.
    if (window.craftMobile?.active()) return;
    last = { y: e.clientY, target: e.target };
    sync();
  }, { passive: true, signal: ac.signal });

  on(document, 'focusin', e => {
    last = { y: -1, target: e.target };
    sync();
  }, { signal: ac.signal });
  window.craftAutoNav = { disable() { ac.abort(); clearTimeout(timer); timer = null; } };
})();

## Compact Monaco Line Height

Aligns Solveit's native Monaco editors with `--craft-code` and `--craft-code-lh` so switching between inline CM6 and Monaco preserves visual density.

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Compact Monaco line height ──────────────────────────────────── */
/* Code size and family come from CRAFT.css. Monaco line-height matches the
   code size; remeasure after webfonts load to keep cursor columns aligned. */
(() => {
  window.craftCompactMonaco?.disable?.();
  let tries = 0, retryTimer = null, createDisp = null, disabled = false;
  const go = () => {
    const mon = window.monaco?.editor;
    if (disabled) return;
    if (!mon) { if (++tries < 40) retryTimer = setTimeout(go, 250); return; }
    // Recompute character widths after the webfonts replace the fallback.
    if (document.fonts) Promise.all([
      document.fonts.load('12px "Craft Iosevka"'),
      document.fonts.load('italic 12px "Craft Iosevka"'),
      document.fonts.load('bold 12px "Craft Iosevka"'),
      document.fonts.load('italic bold 12px "Craft Iosevka"')
    ]).then(() => { mon.remeasureFonts(); mon.getEditors().forEach(ed => ed.layout()); })
      .catch(err => console.warn('Iosevka could not load; using monospace fallback.', err));
    const optsFor = fallbackFs => {
      const cs = getComputedStyle(document.documentElement);
      const fs = parseFloat(cs.getPropertyValue('--craft-code')) || fallbackFs;
      const lh = Math.round(fs * (parseFloat(cs.getPropertyValue('--craft-code-lh')) || 1.2));
      return { fontSize: fs, lineHeight: lh, fontFamily: cs.getPropertyValue('--craft-code-family').trim() || 'monospace' };
    };
    const compact = ed => { try { ed.updateOptions(optsFor(ed.getOption(mon.EditorOption.fontSize))); } catch (e) {} };
    createDisp = mon.onDidCreateEditor?.(compact) || null;
    mon.getEditors().forEach(compact);
    try { mon.getDiffEditors().forEach(d => d.updateOptions(optsFor(12))); } catch (e) {}
  };
  window.craftCompactMonaco = {
    disable() {
      disabled = true;
      clearTimeout(retryTimer);
      retryTimer = null;
      createDisp?.dispose?.();
      createDisp = null;
    },
  };
  go();
})();

## Command & File Launcher (`Mod+Shift+P`)

Searchable palette combining registered `solveitHotkeys` actions with folders and dialogs already loaded by the file tree.

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Launcher (⌘⇧P) ──────────────────────────────────────────────────
   One fuzzy box over commands (every hotkey-registry action, custom and
   Solveit built-in) and dialogs/files (whatever the file tree has loaded).
   Enter runs or opens the selection; Esc or a click outside closes. */
(() => {
  window.craftLauncher?.disable?.();
  const {el, mountTpl, qsa, on, notifyErr} = window.__craftCMCore;
  let root, input, list, results = [], sel = 0, pool = [], prevFocus;

  function score(q, text) {
    if (!q) return 1;
    const t = text.toLowerCase();
    let i = 0, s = 0, run = 0, last = -2;
    for (const ch of q) {
      const j = t.indexOf(ch, i);
      if (j < 0) return 0;
      run = j === last + 1 ? run + 1 : 0;
      s += 1 + run * 2 + (j === 0 || /[\s/_.-]/.test(t[j - 1]) ? 3 : 0);
      last = j; i = j + 1;
    }
    return s - t.length * 0.01;
  }

  const where = n => { try { const u = new URL(n.href, location.href); return u.searchParams.get('name') || u.searchParams.get('fp') || decodeURIComponent(u.pathname); } catch { return ''; } };

  function build() {
    ({root, input, list} = mountTpl('launcher_root', {refs: {input: 'input', list: 'ul'}}));
    on(root, 'mousedown', e => { if (e.target === root) close(); });
    on(input, {
      input: () => { sel = 0; render(); },
      keydown: e => {
        if (e.key === 'ArrowDown' || e.key === 'ArrowUp') { e.preventDefault(); e.stopPropagation(); sel = (sel + (e.key === 'ArrowDown' ? 1 : -1) + results.length) % Math.max(results.length, 1); paint(); }
        else if (e.key === 'Enter') { e.preventDefault(); e.stopPropagation(); run(results[sel]); }
        else if (e.key === 'Escape') { e.preventDefault(); e.stopPropagation(); close(); }
      },
    });
    on(list, 'click', e => { const li = e.target.closest('li[data-i]'); if (li) run(results[+li.dataset.i]); });
    document.body.append(root);
  }

  function render() {
    const q = input.value.trim().toLowerCase();
    results = pool.map(it => ({it, s: score(q, it.label + ' ' + it.detail)})).filter(r => r.s > 0).sort((a, b) => b.s - a.s).slice(0, 60).map(r => r.it);
    sel = Math.min(sel, Math.max(results.length - 1, 0));
    list.replaceChildren(...(results.length ? results.map((it, i) =>
      el('li', {cls: 'kind-' + it.kind, role: 'option', dataset: {i}},
        el('span', {cls: 'label'}, it.label), el('span', {cls: 'detail'}, it.detail), el('span', {cls: 'hint'}, it.hint))
    ) : [el('li', {cls: 'empty'}, 'No matches')]));
    paint();
  }

  function paint() {
    qsa('li[data-i]', list).forEach(li => { const on = +li.dataset.i === sel; li.setAttribute('aria-selected', String(on)); if (on) li.scrollIntoView({block: 'nearest'}); });
  }

  async function open() {
    if (!root) build();
    prevFocus = document.activeElement;
    const hk = window.solveitHotkeys;
    const cmds = (hk?.list?.() || []).filter(a => a.id !== 'launcher').map(a => ({kind: 'command', label: a.label, detail: a.group || '', hint: a.combo ? hk.formatCombo(a.combo) : '', go: () => hk.run(a.id)}));
    pool = cmds; root.classList.add('open'); input.value = ''; sel = 0; render(); input.focus();
    try {
      const entries = await window.solveitFileTree?.entries?.() || [];
      pool = cmds.concat(entries.map(n => ({kind: n.kind, label: n.name, detail: where(n), hint: n.kind, go: () => window.solveitFileTree.open(n)})));
      if (root.classList.contains('open')) render();
    } catch {}
  }

  function close() { if (!root?.classList.contains('open')) return; root.classList.remove('open'); prevFocus?.focus?.(); }
  function run(item) { if (!item) return; close(); setTimeout(() => Promise.resolve().then(item.go).catch(err => { console.error('Launcher action failed', err); notifyErr('Action failed: ' + (err.message || err), 'craftLauncher'); }), 0); }

  window.__hotkeyRegister?.({id: 'launcher', label: 'Open launcher', group: 'Launcher', combo: 'Mod+Shift+P', handler: () => { if (root?.classList.contains('open')) close(); else open(); return true; }});
  window.craftLauncher = { open, close, disable() { close(); root?.remove(); root = input = list = null; } };
})();

## Idle Bottom Composer (`window.craftIdleComposer`)

Hides the bottom Monaco composer when empty and idle; `Shift+I` reveals it, and non-empty drafts stay visible. Set `window.CRAFT_IDLE_COMPOSER = false` before init to opt out.

In [ ]:
#| export_js CRAFT.js
#| eval: false
(() => {
  window.craftIdleComposer?.disable();
  if (window.CRAFT_IDLE_COMPOSER === false) return;
  const {qs, on, onDomChange, MODAL_OPEN_SEL} = window.__craftCMCore;
  let editor, subscriptions = [], requested = false, pending = false;
  const panel = () => document.getElementById('full_editor');
  const temporary = () => editor?.getModel()?.uri.path.endsWith('/temp');
  const hasDraft = () => !!window.edValue?.();
  const reveal = () => {
    requested = true;
    panel()?.classList.remove('craft-composer-idle');
    editor?.layout();
  };
  const unbind = () => {
    subscriptions.forEach(s => s.dispose());
    subscriptions = [];
  };
  const sync = () => {
    pending = false;
    if (window.editor !== editor) {
      unbind();
      editor = window.editor;
      if (editor) {
        subscriptions.push(editor.onDidChangeModel(schedule));
        subscriptions.push(editor.onDidChangeModelContent(schedule));
      }
    }
    const root = panel();
    if (!root) return;
    const editingCell = Boolean(editor?.getModel() && !temporary()) || Boolean(qs('.editable[data-editing="true"]'));
    const emptyDialog = !qs('#dialog-container > .editable');
    const active = requested || (!insertionHolds.size && (emptyDialog || editingCell || hasDraft()));
    const idle = !active;
    if (root.classList.contains('craft-composer-idle') === idle) return;
    root.classList.toggle('craft-composer-idle', idle);
    if (active && editor) editor.layout();
  };
  const schedule = () => { if (!pending) { pending = true; queueMicrotask(sync); } };
  const tabClick = e => {
    const tab = e.target.closest('#full_editor .tabh');
    if (!tab) return;
    if (tab.id === 'tabh-code' && window.craftC2?.shouldInterceptCodeTab?.()) return;
    reveal();
  };
  let shortcutOpened = false, dismissTimer;
  const insertionHolds = new Set(), rootEl = document.documentElement;
  const clearInsertionHolds = () => { insertionHolds.clear(); rootEl.removeAttribute('data-craft-inserting'); };
  const holdForInsertion = () => {
    const token = Symbol();
    insertionHolds.add(token); rootEl.setAttribute('data-craft-inserting', '');
    return () => {
      if (!insertionHolds.delete(token) || insertionHolds.size) return;
      if (!hasDraft() && temporary()) hide();
      rootEl.removeAttribute('data-craft-inserting');
    };
  };
  const hide = () => {
    clearTimeout(dismissTimer);
    shortcutOpened = false;
    requested = false;
    sync();
  };
  const show = () => {
    if (window.editor !== editor) sync();
    const root = panel();
    if (!root || !editor?.getModel()) return false;
    clearTimeout(dismissTimer);
    clearInsertionHolds();
    shortcutOpened ||= root.classList.contains('craft-composer-idle');
    reveal();
    editor.focus();
    return true;
  };
  const dismissShortcut = e => {
    if (!shortcutOpened || e.key !== 'Escape' || e.isComposing || !panel()?.contains(e.target)) return;
    clearTimeout(dismissTimer);
    dismissTimer = setTimeout(() => {
      shortcutOpened = false;
      if (!temporary() || hasDraft() || qs('.editable[data-editing="true"]')) return;
      requested = false;
      sync();
    }, 0);
  };
  const ac = new AbortController(), sig = {signal: ac.signal}, cap = {capture: true, signal: ac.signal};
  const offDom = onDomChange(schedule, {attrs: ['data-editing'], delay: 0});
  const settleEvent = ['htmx', 'after', 'settle'].join(window.htmx?.config?.metaCharacter || ':');
  on(window, 'keydown', dismissShortcut, cap);
  on(document, 'click', tabClick, cap);
  on(document, settleEvent, schedule, sig);
  window.craftIdleComposer = {
    reveal,
    hide,
    show,
    holdForInsertion,
    disable() {
      offDom();
      ac.abort();
      clearTimeout(dismissTimer);
      clearInsertionHolds();
      if (window.craftIdleComposer?.show === show) delete window.craftIdleComposer;
      unbind();
      panel()?.classList.remove('craft-composer-idle');
      editor?.layout();
    }
  };
  sync();
  window.__hotkeyRegister?.({id: 'composer-show', label: 'Show cell editor', group: 'Editor', combo: 'Shift+I', handler: e => {
    if (e?.isComposing || e?.target?.closest?.(`#solveit-file-tree-panel, #solveit-temp-editor, [role="dialog"], ${MODAL_OPEN_SEL}`)) return false;
    return window.craftIdleComposer?.show() === true;
  }});
})();

## Accessibility Pass (`window.craftA11y`)

Mirrors Solveit `uk-tooltip` attributes into `aria-label`s on icon-only buttons, `#find-box` controls, and navbar controls so screen readers and keyboard users get meaningful labels.

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Accessibility: mirror uk-tooltip & fallback aria-labels on icon controls ── */
(() => {
  window.craftA11y?.disable?.();
  const extractTip = raw => {
    if (!raw) return '';
    const s = String(raw).trim();
    const m = s.match(/(?:^|;)\s*title\s*:\s*([^;]+)/i);
    return (m ? m[1] : s).replace(/<[^>]*>/g, '').trim();
  };
  const labelControls = (root = document) => {
    // 1. Any interactive element carrying uk-tooltip / data-uk-tooltip without an accessible name
    for (const el of root.querySelectorAll('[uk-tooltip], [data-uk-tooltip]')) {
      if (!el.matches('button, a, input, select, textarea')) continue;
      if (el.hasAttribute('aria-label') || el.hasAttribute('aria-labelledby')) continue;
      const tip = extractTip(el.getAttribute('uk-tooltip') || el.getAttribute('data-uk-tooltip'));
      if (tip) el.setAttribute('aria-label', tip);
    }
    // 2. #find-box controls (checkboxes wrapped in <label uk-tooltip>, search/replace inputs)
    const fb = root.getElementById ? root.getElementById('find-box') : root.querySelector?.('#find-box');
    if (fb) {
      for (const el of fb.querySelectorAll('input, button, select')) {
        if (el.hasAttribute('aria-label') || el.hasAttribute('aria-labelledby')) continue;
        const lbl = el.closest('label') || (el.id && fb.querySelector(`label[for="${CSS.escape(el.id)}"]`));
        const tip = extractTip(
          el.getAttribute('uk-tooltip') || el.getAttribute('data-uk-tooltip') ||
          lbl?.getAttribute('uk-tooltip') || lbl?.getAttribute('data-uk-tooltip') ||
          el.getAttribute('title') || lbl?.getAttribute('title')
        );
        const fallback = tip || (el.getAttribute('placeholder') || '').replace(/\.+$/, '').trim();
        if (fallback) el.setAttribute('aria-label', fallback);
      }
    }
    // 3. Navbar <select name="mode"> and Dialogs link
    const nav = root.querySelector?.('.uk-container > nav.monster-navbar') || document.querySelector('.uk-container > nav.monster-navbar');
    if (nav) {
      const modeSel = nav.querySelector('select[name="mode"]');
      if (modeSel && !modeSel.hasAttribute('aria-label')) modeSel.setAttribute('aria-label', 'AI mode');
      const dlgsLink = nav.querySelector('a[href^="/?at="]');
      if (dlgsLink && !dlgsLink.hasAttribute('aria-label') && !(dlgsLink.textContent || '').trim()) {
        dlgsLink.setAttribute('aria-label', 'Dialogs');
      }
      // 4. Any remaining icon-only navbar buttons/links without accessible name
      for (const btn of nav.querySelectorAll('button, a')) {
        if (btn.hasAttribute('aria-label') || btn.hasAttribute('aria-labelledby')) continue;
        if ((btn.textContent || '').trim()) continue;
        const tip = extractTip(
          btn.getAttribute('title') || btn.getAttribute('uk-tooltip') || btn.getAttribute('data-uk-tooltip')
        );
        const iconRef = btn.querySelector('use')?.getAttribute('href')?.replace(/^#lc-/, '') ||
                        btn.getAttribute('uk-icon') || btn.querySelector('[uk-icon]')?.getAttribute('uk-icon') || '';
        const hxPost = (btn.getAttribute('hx-post') || btn.getAttribute('hx-get') || '').replace(/^\/|_+$/g, '').replace(/_/g, ' ');
        const fallback = tip || (iconRef ? iconRef.replace(/[-_]+/g, ' ').replace(/^\w/, c => c.toUpperCase()) : '') || hxPost;
        if (fallback) btn.setAttribute('aria-label', fallback);
      }
    }
  };
  let offDom = null;
  const start = () => {
    labelControls(document);
    offDom = window.__craftCMCore.onDomChange(() => labelControls(document), {delay: 120});
  };
  if (document.readyState === 'loading') document.addEventListener('DOMContentLoaded', start);
  else start();
  window.craftA11y = { refresh: () => labelControls(document), disable() { offDom?.(); offDom = null; document.removeEventListener('DOMContentLoaded', start); } };
})();

## Dialog Chrome CSS

Styles the auto-hiding navbar, top-right button reservation, flush bottom composer, `#solveit-launcher`, and `#craft-mobile-toggle`.

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* Reserve top-right space inside desktop top nav so #craft-insert-preview and #craft-mobile-toggle never overlap navbar buttons. */
html:not(.craft-mobile) .uk-container > nav.monster-navbar {
  padding-right: 78px !important;
  box-sizing: border-box;
}

/* ── 7. Top nav: auto-hiding overlay, floating controls ─────────────── */
.uk-container > nav.monster-navbar {
  position: absolute !important;         /* out of flow: scroller keeps the 44px */
  top: 0; left: 0; right: 0;
  z-index: 40;
  height: 31px !important;               /* 0.7 x original */
  min-height: 0 !important;
  flex-wrap: nowrap !important;
  padding-block: 0 !important;
  background: transparent !important;
  border: 0 !important;
  opacity: 0;
  pointer-events: none;                  /* hidden strip never eats clicks */
  transition: opacity 0.18s ease;
}
.uk-container > nav.monster-navbar.nav-shown,
.uk-container > nav.monster-navbar:focus-within { opacity: 1; pointer-events: auto; }
.uk-container > nav.monster-navbar > * { pointer-events: none; }
.uk-container > nav.monster-navbar.nav-shown > *,
.uk-container > nav.monster-navbar:focus-within > * { pointer-events: auto; }

.uk-container > nav.monster-navbar h2 { font-size: 1.05rem !important; line-height: 1.2 !important; }
.uk-container > nav.monster-navbar span.rounded-full.size-4,
.uk-container > nav.monster-navbar a:has(> h1),
.uk-container > nav.monster-navbar h2#dlg-name { display: none !important; }

/* Strip action wrapper, chips (UIkit paints them as outlines), and <select> frame */
.uk-container > nav.monster-navbar :is(> div.ml-auto, .uk-btn, .uk-btn-group, .uk-btn-group > *, select, input:checked + label.uk-btn, .uk-btn.uk-active) {
  background: transparent !important;
  border: 0 !important;
  border-radius: 0 !important;
  box-shadow: none !important;
  outline: 0 !important;
  padding: 0 !important;
}
.uk-container > nav.monster-navbar :is(.uk-btn, select) { color: var(--craft-fg) !important; }
.uk-container > nav.monster-navbar select { padding-inline: 2px !important; }
.uk-container > nav.monster-navbar .uk-btn:hover { color: var(--craft-fg-strong) !important; }
.uk-container > nav.monster-navbar :is(label, button):has(input:checked) {
  color: var(--craft-accent) !important;
  filter: drop-shadow(0 0 4px #7aa2f7cc) drop-shadow(0 0 10px #7aa2f766);
}
.uk-container > nav.monster-navbar .uk-btn.uk-active { color: var(--craft-accent) !important; }
.uk-container > nav.monster-navbar select option { background: var(--craft-recessed) !important; color: var(--craft-fg) !important; }
.uk-container > nav.monster-navbar :is(svg, select, h2) { filter: drop-shadow(0 1px 2px var(--craft-recessed)) drop-shadow(0 0 4px var(--craft-recessed)); }

/* ── Launcher (⌘⇧P) ─────────────────────────────────────────────────── */
#solveit-launcher .solveit-launcher-card { width: min(640px, 92vw); background: var(--craft-bg); border-radius: 8px; overflow: hidden; }
#solveit-launcher input { box-sizing: border-box; width: 100%; padding: 10px 14px; border: 0; border-bottom: 1px solid #ffffff14; outline: 0; background: transparent; color: var(--craft-fg); font-size: 14px; }
#solveit-launcher ul { max-height: 50vh; overflow-y: auto; margin: 0; padding: 4px; list-style: none; }
#solveit-launcher li { display: flex; align-items: baseline; gap: 10px; padding: 5px 10px; border-radius: 4px; color: var(--craft-fg); font-size: 12.5px; cursor: pointer; }
#solveit-launcher li[aria-selected="true"] { background: var(--craft-hover); }
#solveit-launcher li .label { flex: 0 1 auto; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
#solveit-launcher li .detail { flex: 1 1 0%; min-width: 0; color: var(--craft-muted-soft); white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
#solveit-launcher li .hint { flex: 0 0 auto; color: var(--craft-muted); font: 11px monospace; }
#solveit-launcher li.kind-dialog .hint { color: var(--craft-purple); }
#solveit-launcher li.empty { color: var(--craft-muted-soft); cursor: default; }

/* ── Shared floating top-right nav icon buttons (#craft-mobile-toggle & #craft-insert-preview) ── */
.craft-nav-icon-btn {
  position: fixed; z-index: 955; width: 28px; height: 26px; display: grid; place-items: center;
  border: 0; border-radius: 4px; background: transparent; color: #9b9cab; opacity: .4; padding: 0;
  cursor: pointer; transition: opacity .15s ease, color .15s ease;
}
.craft-nav-icon-btn:is(:hover, :focus-visible),
body:has(nav.monster-navbar.nav-shown) .craft-nav-icon-btn { opacity: 1; color: #e0e0e8; }
.craft-nav-icon-btn:focus-visible { outline: 1px solid var(--craft-accent); outline-offset: 2px; }

Verify `05_dialog_chrome.ipynb` exports its templates, JS modules, and CSS cleanly.

In [ ]:
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets

assets = extract_nb_assets(REPO_ROOT/'nbs/craft/05_dialog_chrome.ipynb')
js = '\n'.join(code for _, code in assets['CRAFT.js'])
for sym in ('window.craftMobile', 'window.craftIdleComposer', 'window.craftA11y', 'solveit-launcher'):
    assert sym in js, f"Missing {sym}"
assert '#solveit-launcher' in '\n'.join(code for _, code in assets['CRAFT.css'])

## Live `fastcdp` Verification

In [ ]:
#| live
from fastcdp import Rungs
from craft_dev.cdp import open_solveit

async with open_solveit() as (page, _, page_errors):
    rungs = Rungs(page)

    async with rungs('command launcher open, filter, close'):
        await page.press('P', mod=True, shift=True)
        await page.wait_for_selector('#solveit-launcher.open input', timeout=3)
        await page.fill_text('#solveit-launcher.open input', 'line')
        await page.wait_for("document.querySelectorAll('#solveit-launcher.open ul[role=\"listbox\"] li').length >= 1", timeout=3)
        await page.press('Escape')
        await page.wait_for_selector('#solveit-launcher.open', present=False, timeout=3)

    async with rungs('desktop/mobile mode toggle'):
        assert await page.eval("window.craftMobile.active() === false")
        assert await page.sel_exists('#craft-mobile-toggle')
        await page.eval("window.craftMobile.toggle()")
        assert await page.eval("window.craftMobile.active() === true")
        await page.eval("window.craftMobile.setMode('auto')")
        assert await page.eval("window.craftMobile.active() === false")

    assert not page_errors, f"Uncaught page errors: {page_errors}"